### Testing inference Hugging Face

In [1]:
import torch
import os
from transformers import AutoTokenizer
from src.models import ModelForResidueClassification
from esm.tokenization.sequence_tokenizer import EsmSequenceTokenizer


In [5]:
from src.dataset_class import ResidueInterfaceDataset
from src.data_utils import load_biodl_dataset
from torch.utils.data import DataLoader
from sklearn.metrics import matthews_corrcoef, accuracy_score, f1_score
from tqdm import tqdm
import numpy as np

In [6]:

repo_id = "YuriGardinazzi/PPI-Reps"

# This automatically downloads the weights and config from HF!
model = ModelForResidueClassification.from_pretrained(repo_id)
tokenizer = EsmSequenceTokenizer()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

In [7]:
def run_inference(model, dataloader, seqs, device, max_length=1024):
    """
    Runs inference over a dataloader.
    Returns:
      - all_preds_str: list of binary prediction strings (one per sequence)
      - all_probs:     list of per-residue probability arrays (one per sequence)

    max_length must match the value used in ResidueInterfaceDataset (default 1024).
    Sequences longer than max_length are silently truncated by the dataset, so we
    cap the extraction window here to avoid indexing beyond valid logits.
    The usable residue window is [1 : max_length-1] (positions 0 and max_length-1
    are reserved for BOS and EOS tokens respectively).
    """
    all_preds_str = []
    all_probs = []
    max_residues = max_length - 2  # exclude BOS (pos 0) and EOS (pos max_length-1)

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Processing"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            # Note: attention_mask is unused by ESM3 inside the model but
            # is required by the non-ESM3 branch — always safe to pass.

            outputs = model(input_ids, attention_mask)
            logits = outputs["logits"]

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int().cpu().numpy()
            probs_np = probs.cpu().numpy()

            batch_start = len(all_preds_str)
            for i in range(len(preds)):
                global_idx = batch_start + i
                original_len = len(seqs[global_idx])

                # Cap to the actual number of residues the dataset encoded.
                # Sequences longer than max_length-2 were truncated at dataset time,
                # so we must not try to read more positions than were encoded.
                effective_len = min(original_len, max_residues)

                # Skip BOS token (index 0), extract exactly effective_len residues
                start_idx = 1
                end_idx = start_idx + effective_len

                valid_preds = preds[i][start_idx:end_idx]
                valid_probs = probs_np[i][start_idx:end_idx]

                if original_len > max_residues:
                    print(f"⚠️  Sequence {global_idx} (len={original_len}) was truncated to {max_residues} residues.")

                all_preds_str.append("".join(map(str, valid_preds)))
                all_probs.append(valid_probs)

    return all_preds_str, all_probs

In [8]:
TEST_CSV_PATH = "data/final_zk448_test.csv"
DATASET_TYPE = "p"
TEST_OUTPUT_FILE = "test_set_predictions.csv"
BATCH_SIZE = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
test_seqs, test_labels = load_biodl_dataset(TEST_CSV_PATH, dataset_type=DATASET_TYPE)

test_dataset = ResidueInterfaceDataset(test_seqs, test_labels, tokenizer)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------------------------
# C. Run inference
# -----------------------------------------------
print("🔹 Running inference on test set...")
test_preds_str, test_probs = run_inference(model, test_dataloader, test_seqs, DEVICE)

# -----------------------------------------------
# D. Compute metrics (flattened over all residues)
# -----------------------------------------------
MAX_RESIDUES = 1024 - 2  # must match max_length used in dataset (1024) minus BOS and EOS

all_true = []
all_pred = []

for i, (pred_str, true_labels) in enumerate(zip(test_preds_str, test_labels)):
    pred_arr = np.array(list(pred_str), dtype=int)
    # Truncate true labels to match what the dataset actually encoded
    true_arr = np.array(true_labels[:MAX_RESIDUES], dtype=int)

    # Sanity check: lengths must match after truncation
    if len(pred_arr) != len(true_arr):
        print(f"⚠️  Length mismatch at sequence {i}: pred={len(pred_arr)}, true={len(true_arr)}. Skipping.")
        continue

    all_true.extend(true_arr.tolist())
    all_pred.extend(pred_arr.tolist())

all_true = np.array(all_true)
all_pred = np.array(all_pred)

accuracy = accuracy_score(all_true, all_pred)
mcc      = matthews_corrcoef(all_true, all_pred)
f1       = f1_score(all_true, all_pred, zero_division=0)
f1_macro = f1_score(all_true, all_pred, average="macro", zero_division=0)

print("\n" + "="*45)
print("          TEST SET METRICS (residue-level)")
print("="*45)
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  MCC       : {mcc:.4f}")
print(f"  F1 (pos)  : {f1:.4f}")
print(f"  F1 (macro): {f1_macro:.4f}")
print(f"  Total residues evaluated: {len(all_true)}")
print("="*45)

# -----------------------------------------------
# E. Save per-sequence results to CSV
# -----------------------------------------------
per_seq_rows = []
for i, (seq, pred_str, true_labels, probs_arr) in enumerate(
    zip(test_seqs, test_preds_str, test_labels, test_probs)
):
    pred_arr = np.array(list(pred_str), dtype=int)
    true_arr = np.array(true_labels[:MAX_RESIDUES], dtype=int)  # match truncation

    if len(pred_arr) != len(true_arr):
        continue  # already warned above

    seq_acc = accuracy_score(true_arr, pred_arr)
    seq_mcc = matthews_corrcoef(true_arr, pred_arr) if len(np.unique(true_arr)) > 1 else float("nan")
    seq_f1  = f1_score(true_arr, pred_arr, zero_division=0)

    per_seq_rows.append({
        "sequence":       seq[:MAX_RESIDUES],  # show only the evaluated portion
        "true_labels":    "".join(map(str, true_arr)),
        "prediction":     pred_str,
        "seq_accuracy":   round(seq_acc, 4),
        "seq_mcc":        round(seq_mcc, 4) if not np.isnan(seq_mcc) else "N/A",
        "seq_f1":         round(seq_f1, 4),
    })

#results_df = pd.DataFrame(per_seq_rows)
#results_df.to_csv(TEST_OUTPUT_FILE, index=False)
#print(f"\n✅ Per-sequence results saved to {TEST_OUTPUT_FILE}")

/orfeo/LTS/LADE/LT_storage/bio_data/ppi/PPI-Reps/finetuning/src/data_utils.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: str(x).replace(",", "").strip())


🔹 Running inference on test set...


Processing:   0%|          | 0/336 [00:00<?, ?it/s]

Processing: 100%|██████████| 336/336 [00:39<00:00,  8.56it/s]



          TEST SET METRICS (residue-level)
  Accuracy  : 0.8291
  MCC       : 0.5388
  F1 (pos)  : 0.6282
  F1 (macro): 0.7586
  Total residues evaluated: 84941
